In [11]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
import re
from typing import List, Dict, Tuple
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# CAREER RECOMMENDATION SYSTEM (FIXED VERSION)
# ============================================================================

class CareerRecommendationSystem:
    """
    A content-based recommendation system that matches student skills 
    and career interests with relevant job opportunities from alumni data.
    
    Uses TF-IDF vectorization and cosine similarity for retrieval.
    """
    
    def __init__(self, max_features=5000, ngram_range=(1, 2), min_df=2):
        """
        Initialize the recommendation system.
        
        Args:
            max_features: Maximum number of features for TF-IDF
            ngram_range: Range of n-grams to extract
            min_df: Minimum document frequency for terms
        """
        self.vectorizer = TfidfVectorizer(
            ngram_range=ngram_range,
            max_features=max_features,
            min_df=min_df,
            stop_words='english',
            lowercase=True
        )
        self.job_vectors = None
        self.job_data = None
        self.is_fitted = False
    
    def _create_text_features(self, job_df: pd.DataFrame) -> pd.Series:
        """
        Combine relevant job features into a single text representation.
        
        Args:
            job_df: DataFrame containing job information
            
        Returns:
            Series of combined text features
        """
        # Combine multiple fields with appropriate weighting
        text_features = (
            job_df['job_title'].fillna('') + " " +
            job_df['job_title'].fillna('') + " " +  # Double weight for title
            job_df['required_skills'].fillna('') + " " +
            job_df['required_skills'].fillna('') + " " +  # Double weight for skills
            job_df.get('industry', '').fillna('') + " " +
            job_df.get('job_description_length', '').astype(str)
        ).str.lower()
        
        return text_features
    
    def fit(self, job_df: pd.DataFrame):
        """
        Build the searchable index from job postings.
        
        Args:
            job_df: DataFrame with columns: job_id, job_title, company_name, 
                    required_skills, and optionally description, category
        
        Returns:
            self
        """
        # Validate required columns
        required_cols = ['job_title', 'required_skills']
        missing_cols = [col for col in required_cols if col not in job_df.columns]
        if missing_cols:
            raise ValueError(f"Missing required columns: {missing_cols}")
        
        # Store job data
        self.job_data = job_df.copy().reset_index(drop=True)
        
        # Create text features and vectorize
        text_features = self._create_text_features(job_df)
        self.job_vectors = self.vectorizer.fit_transform(text_features)
        
        self.is_fitted = True
        print(f"✓ System fitted on {len(job_df)} job postings")
        print(f"✓ Vocabulary size: {len(self.vectorizer.vocabulary_)}")
        print(f"✓ Unique job titles: {job_df['job_title'].nunique()}")
        
        return self
    
    def recommend(self, user_skills: str, desired_role: str = "", 
                  top_k: int = 5, min_score: float = 0.0, 
                  return_job_ids: bool = False) -> pd.DataFrame:
        """
        Find best matching jobs for a user profile.
        
        Args:
            user_skills: Comma-separated string of user's skills
            desired_role: Target job role/title (optional)
            top_k: Number of recommendations to return
            min_score: Minimum relevance score threshold (0-100)
            return_job_ids: Whether to include job_id in output
            
        Returns:
            DataFrame with recommended jobs and relevance scores
        """
        if not self.is_fitted:
            raise ValueError("System must be fitted before making recommendations")
        
        # Create query by combining role and skills
        query = f"{desired_role} {desired_role} {user_skills}".lower().strip()
        
        # Vectorize query
        query_vec = self.vectorizer.transform([query])
        
        # Compute similarities with all jobs
        similarities = cosine_similarity(query_vec, self.job_vectors)[0]
        
        # Get top-k indices
        top_k = min(top_k, len(similarities))
        top_indices = similarities.argsort()[-top_k:][::-1]
        
        # Create results DataFrame
        results = self.job_data.iloc[top_indices].copy()
        results['relevance_score'] = similarities[top_indices] * 100
        
        # Filter by minimum score
        results = results[results['relevance_score'] >= min_score]
        
        # Select output columns
        output_cols = ['job_title', 'company_name', 'required_skills', 'relevance_score']
        if return_job_ids:
            output_cols.insert(0, 'job_id')
        
        available_cols = [col for col in output_cols if col in results.columns]
        
        return results[available_cols].reset_index(drop=True)
    
    def _normalize_skill(self, skill: str) -> str:
        """Normalize a skill string for comparison."""
        # Remove special chars, lowercase, strip whitespace
        return re.sub(r'[^a-z0-9\s]', '', skill.lower().strip())
    
    def get_skill_gaps(self, user_skills: str, target_job_id: int) -> Dict:
        """
        Identify skill gaps between user and a specific job.
        
        Args:
            user_skills: User's current skills (comma-separated)
            target_job_id: ID of the target job
            
        Returns:
            Dictionary with matching skills and missing skills
        """
        if not self.is_fitted:
            raise ValueError("System must be fitted first")
        
        # Find target job
        target_job = self.job_data[self.job_data['job_id'] == target_job_id]
        if target_job.empty:
            raise ValueError(f"Job ID {target_job_id} not found")
        
        # Parse and normalize skills
        user_skill_list = [s.strip() for s in user_skills.split(',')]
        user_skill_set = set([self._normalize_skill(s) for s in user_skill_list])
        
        required_skills = target_job.iloc[0]['required_skills']
        required_skill_list = [s.strip() for s in str(required_skills).split(',')]
        required_skill_set = set([self._normalize_skill(s) for s in required_skill_list])
        
        # Remove empty strings
        user_skill_set.discard('')
        required_skill_set.discard('')
        
        matching = user_skill_set & required_skill_set
        missing = required_skill_set - user_skill_set
        extra = user_skill_set - required_skill_set
        
        return {
            'job_title': target_job.iloc[0]['job_title'],
            'company': target_job.iloc[0].get('company_name', 'Unknown'),
            'matching_skills': sorted(list(matching)),
            'missing_skills': sorted(list(missing)),
            'additional_skills': sorted(list(extra)),
            'match_percentage': (len(matching) / len(required_skill_set) * 100 
                               if required_skill_set else 0)
        }


def honest_evaluation(system, train_df, test_cases):
    """
    Honest evaluation that doesn't mislead with inflated metrics.
    """
    
    print("\n" + "="*70)
    print("SYSTEM VALIDATION")
    print("="*70)
    
    # -------------------------------------------------------------------------
    # METRIC 1: DATASET STATISTICS (Context)
    # -------------------------------------------------------------------------
    print("\n📊 Dataset Characteristics:")
    print(f"   Total Jobs: {len(train_df)}")
    print(f"   Unique Titles: {train_df['job_title'].nunique()}")
    print(f"   Avg Jobs per Title: {len(train_df) / train_df['job_title'].nunique():.0f}")
    print(f"   Unique Companies: {train_df['company_name'].nunique()}")
    
    # -------------------------------------------------------------------------
    # METRIC 2: MATCH SCORE DISTRIBUTION (Quality Check)
    # -------------------------------------------------------------------------
    print("\n📊 Match Score Analysis:")
    
    all_scores = []
    for tc in test_cases:
        recs = system.recommend(tc['skills'], tc['role'], top_k=10)
        if len(recs) > 0:
            all_scores.extend(recs['relevance_score'].tolist())
    
    if all_scores:
        print(f"   Average Match Score: {np.mean(all_scores):.1f}%")
        print(f"   Median Match Score: {np.median(all_scores):.1f}%")
        print(f"   Score Range: {min(all_scores):.1f}% - {max(all_scores):.1f}%")
        
        # Interpret scores
        high_quality = sum(1 for s in all_scores if s >= 60)
        medium_quality = sum(1 for s in all_scores if 40 <= s < 60)
        low_quality = sum(1 for s in all_scores if s < 40)
        
        print(f"\n   Quality Distribution:")
        print(f"   - High Match (≥60%): {high_quality}/{len(all_scores)} ({high_quality/len(all_scores)*100:.0f}%)")
        print(f"   - Medium Match (40-60%): {medium_quality}/{len(all_scores)} ({medium_quality/len(all_scores)*100:.0f}%)")
        print(f"   - Low Match (<40%): {low_quality}/{len(all_scores)} ({low_quality/len(all_scores)*100:.0f}%)")
    
    # -------------------------------------------------------------------------
    # METRIC 3: ROLE ACCURACY (Does it find the right job type?)
    # -------------------------------------------------------------------------
    print("\n📊 Role Matching Accuracy:")
    
    role_matches = []
    for tc in test_cases:
        recs = system.recommend(tc['skills'], tc['role'], top_k=5)
        
        if len(recs) > 0:
            # Check if top recommendation matches target role keywords
            target_keywords = tc['role'].lower().split()
            top_title = recs.iloc[0]['job_title'].lower()
            
            # Simple keyword matching
            match = any(keyword in top_title for keyword in target_keywords 
                       if keyword not in ['engineer', 'developer'])
            role_matches.append(match)
            
            status = "✓" if match else "✗"
            print(f"   {status} Query: '{tc['role']}' → Got: '{recs.iloc[0]['job_title']}'")
        else:
            role_matches.append(False)
            print(f"   ✗ Query: '{tc['role']}' → No results")
    
    accuracy = sum(role_matches) / len(role_matches) * 100 if role_matches else 0
    print(f"\n   Role Match Accuracy: {accuracy:.0f}%")
    
    # -------------------------------------------------------------------------
    # METRIC 4: COVERAGE ANALYSIS
    # -------------------------------------------------------------------------
    print("\n📊 System Coverage:")
    
    # Test with diverse queries
    diverse_queries = [
        "Python", "Java", "JavaScript", "R", "SQL",
        "Machine Learning", "Web Development", "Mobile",
        "Cloud", "DevOps", "Data Analysis"
    ]
    
    jobs_found = set()
    queries_with_results = 0
    
    for query in diverse_queries:
        recs = system.recommend(query, top_k=10, return_job_ids=True)
        if len(recs) > 0:
            queries_with_results += 1
            jobs_found.update(recs['job_id'].tolist())
    
    print(f"   Queries with Results: {queries_with_results}/{len(diverse_queries)}")
    print(f"   Unique Jobs Reached: {len(jobs_found)}/{len(train_df)} ({len(jobs_found)/len(train_df)*100:.1f}%)")
    
    # -------------------------------------------------------------------------
    # METRIC 5: DIVERSITY CHECK
    # -------------------------------------------------------------------------
    print("\n📊 Recommendation Diversity:")
    
    all_titles = []
    for tc in test_cases:
        recs = system.recommend(tc['skills'], tc['role'], top_k=5)
        all_titles.extend(recs['job_title'].tolist())
    
    unique_ratio = len(set(all_titles)) / len(all_titles) if all_titles else 0
    print(f"   Unique Title Ratio: {unique_ratio:.2f}")
    print(f"   Total Recommendations: {len(all_titles)}")
    print(f"   Unique Titles: {len(set(all_titles))}")
    
    if unique_ratio < 0.5:
        print("   ⚠️  Low diversity - recommendations are repetitive")
    
    # -------------------------------------------------------------------------
    # FINAL ASSESSMENT
    # -------------------------------------------------------------------------
    print("\n" + "="*70)
    print("SYSTEM ASSESSMENT")
    print("="*70)
    
    print("\n✓ STRENGTHS:")
    if np.mean(all_scores) > 45:
        print("   • Produces relevant matches for AI/Data Science roles")
    if accuracy > 60:
        print("   • Successfully identifies correct job types")
    print("   • Fast retrieval using TF-IDF (no GPU needed)")
    print("   • Provides interpretable match scores")
    
    print("\n⚠️  LIMITATIONS:")
    print("   • Dataset limited to 20 job types (AI/Data focused)")
    print("   • Low coverage - only ~1% of jobs can be recommended")
    print("   • No web development, mobile, or non-tech roles")
    if unique_ratio < 0.5:
        print("   • Recommendations can be repetitive")
    
    print("\n💡 RECOMMENDATIONS:")
    print("   1. Position as 'AI Career Specialization Tool'")
    print("   2. Add filters: experience level, location, salary")
    print("   3. Implement user feedback loop")
    print("   4. Expand dataset to include diverse roles")
    
    print("="*70)


# ============================================================================
# UPDATE YOUR MAIN FUNCTION - REPLACE EVALUATION SECTION
# ============================================================================

def main():
    """
    Main execution pipeline: Load data → Train → Evaluate → Test
    """
    
    print("="*70)
    print("CAREER GUIDANCE PLATFORM - RECOMMENDATION SYSTEM")
    print("="*70)
    
    # [Keep your existing data loading and training code...]
    # ... (lines 1-200 stay the same)
    
    print("\n[1/5] Loading Data...")
    df = pd.read_csv('ai_job_dataset.csv')
    
    if 'job_id' not in df.columns:
        df['job_id'] = range(len(df))
    
    print(f"✓ Loaded {len(df)} job postings")
    
    print("\n[2/5] Analyzing Dataset...")
    
    print("\n[3/5] Splitting Data...")
    try:
        title_counts = df['job_title'].value_counts()
        df['title_group'] = df['job_title'].apply(
            lambda x: x if title_counts[x] >= 10 else 'Other'
        )
        
        train_df, test_df = train_test_split(
            df, test_size=0.2, random_state=42,
            stratify=df['title_group'], shuffle=True
        )
        
        train_df = train_df.drop('title_group', axis=1)
        test_df = test_df.drop('title_group', axis=1)
        print("✓ Used stratified split")
    except:
        train_df, test_df = train_test_split(
            df, test_size=0.2, random_state=42, shuffle=True
        )
    
    print(f"✓ Training: {len(train_df)}, Test: {len(test_df)}")
    
    print("\n[4/5] Training System...")
    system = CareerRecommendationSystem(max_features=5000, ngram_range=(1, 2), min_df=2)
    system.fit(train_df)
    
    # -------------------------------------------------------------------------
    # TEST SCENARIOS
    # -------------------------------------------------------------------------
    print("\n[5/5] Testing with Student Scenarios...")
    print("\n" + "-"*70)
    
    test_cases = [
        {
            "name": "🎓 Final Year CS Student",
            "skills": "Python, Java, SQL, Git",
            "role": "Software Engineer",
            "description": "Entry-level software engineering role"
        },
        {
            "name": "🚀 Data Science Enthusiast",
            "skills": "Python, Machine Learning, Pandas, Statistics",
            "role": "Data Scientist",
            "description": "ML-focused data science position"
        },
        {
            "name": "💼 Business Analyst",
            "skills": "Excel, SQL, PowerBI, Python",
            "role": "Business Analyst",
            "description": "Business intelligence role"
        },
        {
            "name": "🔧 DevOps Engineer",
            "skills": "Linux, Docker, Kubernetes, AWS, Python",
            "role": "DevOps Engineer",
            "description": "Infrastructure automation role"
        }
    ]
    
    for tc in test_cases:
        print(f"\n{tc['name']}")
        print(f"Skills: {tc['skills']}")
        print(f"Target: {tc['role']}")
        print("\nTop 3 Recommendations:")
        
        recs = system.recommend(tc['skills'], tc['role'], top_k=3)
        
        if len(recs) == 0:
            print("  ⚠️  No matches found")
        else:
            for idx, (_, row) in enumerate(recs.iterrows(), 1):
                print(f"  {idx}. {row['job_title']} @ {row['company_name']}")
                print(f"     Match: {row['relevance_score']:.1f}% | Skills: {row['required_skills'][:60]}...")
        
        print("-"*70)
    
    # -------------------------------------------------------------------------
    # SKILL GAP DEMO
    # -------------------------------------------------------------------------
    print("\n" + "="*70)
    print("SKILL GAP ANALYSIS")
    print("="*70)
    
    sample_job = train_df.iloc[0]
    user_skills = "Python, SQL"
    
    print(f"\nStudent Skills: {user_skills}")
    print(f"Target: {sample_job['job_title']} @ {sample_job['company_name']}")
    
    gap = system.get_skill_gaps(user_skills, sample_job['job_id'])
    
    print(f"\n✓ Matching: {', '.join(gap['matching_skills']) or 'None'}")
    print(f"⚠️  Missing: {', '.join(gap['missing_skills']) or 'None'}")
    print(f"📊 Match: {gap['match_percentage']:.0f}%")
    
    # -------------------------------------------------------------------------
    # HONEST EVALUATION (REPLACE THE MISLEADING METRICS)
    # -------------------------------------------------------------------------
    honest_evaluation(system, train_df, test_cases)
    
    print("\n💾 Training Complete!")
    print("\n📌 FOR YOUR PROJECT PRESENTATION:")
    print("   ✓ Emphasize: Skill gap analysis & personalized roadmaps")
    print("   ✓ Emphasize: Fast, interpretable recommendations")
    print("   ✓ Acknowledge: Dataset limited to AI/Data Science domain")
    print("   ✓ Future work: Expand to web dev, mobile, cloud roles")
    
    return system


if __name__ == "__main__":
    system = main()




CAREER GUIDANCE PLATFORM - RECOMMENDATION SYSTEM

[1/5] Loading Data...
✓ Loaded 15000 job postings

[2/5] Analyzing Dataset...

[3/5] Splitting Data...
✓ Used stratified split
✓ Training: 12000, Test: 3000

[4/5] Training System...
✓ System fitted on 12000 job postings
✓ Vocabulary size: 4890
✓ Unique job titles: 20

[5/5] Testing with Student Scenarios...

----------------------------------------------------------------------

🎓 Final Year CS Student
Skills: Python, Java, SQL, Git
Target: Software Engineer

Top 3 Recommendations:
  1. AI Software Engineer @ DataVision Ltd
     Match: 48.5% | Skills: Mathematics, Java, SQL...
  2. AI Software Engineer @ Quantum Computing Inc
     Match: 48.4% | Skills: SQL, Git, Java, Hadoop...
  3. AI Software Engineer @ Quantum Computing Inc
     Match: 47.5% | Skills: Azure, SQL, Git...
----------------------------------------------------------------------

🚀 Data Science Enthusiast
Skills: Python, Machine Learning, Pandas, Statistics
Target: Data 